# Set Environment

In [1]:
## Install dependencies and set up paths.
## The notebook loads data directly from the pipeline so it always
## reflects the most current version of the cleaned dataset.
!pip install vaderSentiment -q
!pip install lingua-language-detector deep-translator -q
!pip install wordcloud scikit-learn -q
!pip install thefuzz python-Levenshtein -q
!pip install textstat -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
import sys
import pandas as pd
import numpy as np
import re
import unicodedata
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from importlib import reload

drive.mount('/content/drive', force_remount=True)

BASE_DIR   = '/content/drive/Shareddrives/essentis_intern_drive'
DATA_DIR   = os.path.join(BASE_DIR, 'data')
SRC_DIR    = os.path.join(DATA_DIR, 'src_reviews_cleaning')
EXPORT_DIR = os.path.join(BASE_DIR, 'charts', 'marketing_quotes')

if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

os.makedirs(EXPORT_DIR, exist_ok=True)

print("Setup complete.")
print("Exports will save to:", EXPORT_DIR)

Mounted at /content/drive
Setup complete.
Exports will save to: /content/drive/Shareddrives/essentis_intern_drive/charts/marketing_quotes


# Load Data

In [3]:
# Load cleaned data directly from the pipeline.
# text_for_analysis uses translated text for non-English reviews
# so all scoring runs on English text only.

import src_reviews_cleaning.pipeline as P
reload(P)

df = P.run(save_csv=False)

# Build the analysis text column
df['text_for_analysis'] = df['review_text_translated'].where(
    df['review_text_translated'].notna(), df['review_text']
)

# Drop rows with no usable text
df = df.dropna(subset=['text_for_analysis']).reset_index(drop=True)

# Score sentiment with VADER
analyzer = SentimentIntensityAnalyzer()
scores = df['text_for_analysis'].apply(
    lambda x: analyzer.polarity_scores(str(x))
)
df['sentiment_compound'] = scores.apply(lambda x: x['compound'])
df['sentiment_label']    = df['sentiment_compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)

print(f"Shape: {df.shape}")
print(f"\nCourse breakdown:")
print(df['course'].value_counts(dropna=False).to_string())
print(f"\nSentiment breakdown:")
print(df['sentiment_label'].value_counts().to_string())

Step 1/4: Loading raw data...
  google: 256 rows
  clean: 541 rows
  new: 49 rows
Step 2/4: Normalizing columns...


/content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/normalizers.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(normalized, ignore_index=True)


  Combined shape: (846, 17)
Step 3/4: Cleaning data (includes language detection + translation)...
  [17/846] Translated from FRENCH: Très bonne école de formation. Les cours sont bien...
  [206/846] Translated from GERMAN: Der KI Kompaktkurs bei WBS Coding School bietet ei...
  [207/846] Translated from GERMAN: Abhängig vom eigenen Skillset und der jeweiligen G...
  [208/846] Translated from GERMAN: Der AI Compact-Kurs war ausgezeichnet. Er hat mir ...
  [209/846] Translated from GERMAN: Von Mai bis Juli 2025 habe ich an der WBS eine Wei...
  [210/846] Translated from GERMAN: Das Bootcamp "AI for Business" war für mich ein ec...
  [211/846] Translated from GERMAN: Ich habe den Data Science-Kurs abgeschlossen und e...
  [212/846] Translated from GERMAN: Anfangs war ich unsicher, ob mein Vorwissen für da...
  [213/846] Translated from GERMAN: Ich bin mit wenig Programmier-Erfahrung ins Data S...
  [214/846] Translated from GERMAN: Ich habe in den letzten 8 Wochen am KI-Kurs der WB...
  

# Quote Extraction

In [4]:
# Core extraction functions
#
# Instead of truncating at a fixed character limit, we now:
#   1. Split the review into individual sentences
#   2. Score each sentence with VADER
#   3. Prioritise sentences that mention getting hired or career outcomes
#   4. Select complete sentences that fit within each length band
#   5. Never cut a sentence mid-way — always return complete sentences
#
# Length bands are now defined by sentence count not character count:
#   short:  1 complete sentence (the single best sentence in the review)
#   medium: 2-3 complete sentences (the best consecutive passage)
#   long:   3-5 complete sentences (a complete thought from anywhere in the review)
#   full:   the entire review text
#
# Hiring and career outcome mentions are always prioritised over pure
# sentiment score when selecting sentences.

import re
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Keywords that indicate a career outcome mention.
# Sentences containing these are prioritised in short and medium quotes
# regardless of their VADER score.
HIRING_KEYWORDS = [
    "got a job", "found a job", "landed a job", "got hired", "been hired",
    "job offer", "employed", "working as", "now work", "currently work",
    "full time job", "started working", "new job", "new role", "new position",
    "career change", "career switch", "changed career", "hired",
    "job search", "interview", "got an offer", "accepted a position",
]


def mentions_hiring(sentence: str) -> bool:
    """
    Check whether a sentence contains any hiring or career outcome keywords.

    Args:
        sentence (str): A single sentence from a review.

    Returns:
        bool: True if any hiring keyword is found in the lowercased sentence.
    """
    lower = sentence.lower()
    return any(kw in lower for kw in HIRING_KEYWORDS)


def clean_text(text: str) -> str:
    """
    Clean review text for display and sentence splitting.
    Collapses whitespace and normalises line breaks without removing content.

    Args:
        text (str): Raw review text.

    Returns:
        str: Cleaned text suitable for sentence splitting and display.
    """
    text = str(text).strip()
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def split_sentences(text: str) -> list:
    """
    Split review text into individual complete sentences.

    Uses punctuation-based splitting while preserving sentence integrity.
    Filters out fragments under 15 characters that are not useful as quotes.
    Handles common abbreviations to avoid false splits on e.g. "Mr." or "Dr."

    Args:
        text (str): Cleaned review text.

    Returns:
        list: List of complete sentence strings.
    """
    # Temporarily protect common abbreviations from being split
    text = re.sub(r'\b(Mr|Mrs|Ms|Dr|Prof|St|vs|etc|approx|incl)\.',
                  r'\1<DOT>', text)

    # Split on sentence-ending punctuation followed by whitespace + capital
    # or end of string
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z\u201C\u2018"])', text)

    # Restore protected dots
    sentences = [s.replace('<DOT>', '.').strip() for s in raw]

    # Filter out very short fragments
    return [s for s in sentences if len(s) >= 15]


def score_sentence(sentence: str) -> float:
    """
    Score a single sentence using VADER compound score.

    Args:
        sentence (str): A single sentence.

    Returns:
        float: Compound sentiment score between -1 and +1.
    """
    return analyzer.polarity_scores(sentence)['compound']


def rank_sentences(sentences: list, sentiment_label: str) -> list:
    """
    Rank sentences by relevance for quote extraction.

    Sentences mentioning hiring or career outcomes are ranked first
    regardless of sentiment score. Remaining sentences are ranked by
    VADER score — highest for positive reviews, lowest for negative.

    Args:
        sentences (list): List of sentence strings from a review.
        sentiment_label (str): "positive" or "negative".

    Returns:
        list: Sentences sorted by relevance, most relevant first.
    """
    hiring   = [s for s in sentences if mentions_hiring(s)]
    non_hiring = [s for s in sentences if not mentions_hiring(s)]

    if sentiment_label == 'positive':
        non_hiring.sort(key=score_sentence, reverse=True)
    else:
        non_hiring.sort(key=score_sentence, reverse=False)

    return hiring + non_hiring


def extract_short_quote(text: str, sentiment_label: str) -> str:
    """
    Extract the single most impactful complete sentence as a short quote.

    Prioritises sentences mentioning hiring or career outcomes.
    Falls back to the highest scoring sentence by VADER.
    Always returns a complete sentence — never truncates mid-sentence.

    Args:
        text (str): Cleaned review text.
        sentiment_label (str): "positive" or "negative".

    Returns:
        str: One complete sentence.
    """
    sentences = split_sentences(text)
    if not sentences:
        return text

    ranked = rank_sentences(sentences, sentiment_label)
    return ranked[0]


def extract_medium_quote(text: str, sentiment_label: str) -> str:
    """
    Extract the best 2-3 complete consecutive sentences as a medium quote.

    First checks whether any hiring-related sentences exist. If so, finds
    the window of 2-3 consecutive sentences that contains the hiring
    sentence and has the highest average sentiment score.
    Otherwise selects the best consecutive 2-3 sentence window by
    average VADER score.
    Always returns complete sentences — never truncates mid-sentence.

    Args:
        text (str): Cleaned review text.
        sentiment_label (str): "positive" or "negative".

    Returns:
        str: 2-3 complete sentences joined with a space.
    """
    sentences = split_sentences(text)
    if len(sentences) == 0:
        return text
    if len(sentences) == 1:
        return sentences[0]
    if len(sentences) == 2:
        return ' '.join(sentences)

    best_window = None
    best_score  = -999 if sentiment_label == 'positive' else 999
    best_has_hiring = False

    for window_size in [2, 3]:
        for i in range(len(sentences) - window_size + 1):
            window      = sentences[i:i + window_size]
            combined    = ' '.join(window)
            has_hiring  = any(mentions_hiring(s) for s in window)
            avg_score   = float(np.mean([score_sentence(s) for s in window]))

            # Always prefer windows containing hiring mentions
            if has_hiring and not best_has_hiring:
                best_window     = combined
                best_score      = avg_score
                best_has_hiring = True
                continue

            if has_hiring == best_has_hiring:
                if sentiment_label == 'positive' and avg_score > best_score:
                    best_score  = avg_score
                    best_window = combined
                elif sentiment_label == 'negative' and avg_score < best_score:
                    best_score  = avg_score
                    best_window = combined

    return best_window if best_window else ' '.join(sentences[:2])


def extract_long_quote(text: str, sentiment_label: str) -> str:
    """
    Extract the best 3-5 complete consecutive sentences as a long quote.

    Uses the same hiring-priority logic as extract_medium_quote but
    over a larger window of 3-5 sentences. This captures a complete
    thought or narrative arc from anywhere in the review — not just
    the beginning or end.
    Always returns complete sentences — never truncates mid-sentence.

    Args:
        text (str): Cleaned review text.
        sentiment_label (str): "positive" or "negative".

    Returns:
        str: 3-5 complete sentences joined with a space.
    """
    sentences = split_sentences(text)
    if len(sentences) <= 3:
        return ' '.join(sentences)

    best_window     = None
    best_score      = -999 if sentiment_label == 'positive' else 999
    best_has_hiring = False

    for window_size in [3, 4, 5]:
        for i in range(len(sentences) - window_size + 1):
            window      = sentences[i:i + window_size]
            combined    = ' '.join(window)
            has_hiring  = any(mentions_hiring(s) for s in window)
            avg_score   = float(np.mean([score_sentence(s) for s in window]))

            if has_hiring and not best_has_hiring:
                best_window     = combined
                best_score      = avg_score
                best_has_hiring = True
                continue

            if has_hiring == best_has_hiring:
                if sentiment_label == 'positive' and avg_score > best_score:
                    best_score  = avg_score
                    best_window = combined
                elif sentiment_label == 'negative' and avg_score < best_score:
                    best_score  = avg_score
                    best_window = combined

    return best_window if best_window else ' '.join(sentences[:3])


def extract_full_quote(text: str) -> str:
    """
    Return the complete review text unchanged.
    No truncation — the full review is preserved for reference.

    Args:
        text (str): Cleaned review text.

    Returns:
        str: Complete review text.
    """
    return text


print("Quote extraction functions loaded.")
print(f"Hiring keywords tracked: {len(HIRING_KEYWORDS)}")
print("\nFunctions: extract_short_quote, extract_medium_quote, extract_long_quote, extract_full_quote")

# Quick sanity check on a sample review
sample = df['text_for_analysis'].iloc[0]
print(f"\nSample review sentences: {len(split_sentences(clean_text(sample)))}")
print(f"Short quote: {extract_short_quote(clean_text(sample), 'positive')}")

Quote extraction functions loaded.
Hiring keywords tracked: 23

Functions: extract_short_quote, extract_medium_quote, extract_long_quote, extract_full_quote

Sample review sentences: 4
Short quote: a 360 degree turn: The full-stack web development boot camp at WBS COODING SCHOOL was an intense but rewarding journey.


# Version Extraction

In [5]:
# Apply all four extraction functions to every review in the dataset.
# This creates four new columns on df:
#   quote_short   under 100 characters
#   quote_medium  100 to 250 characters
#   quote_long    250 to 500 characters
#   quote_full    complete text up to 1000 characters
#
# All quotes are extracted using sentiment-aware selection so the most
# impactful sentence or passage is always chosen for each review.

print("Cleaning text...")
df['text_clean'] = df['text_for_analysis'].apply(clean_text)

print("Extracting short quotes (under 100 chars)...")
df['quote_short'] = df.apply(
    lambda row: extract_short_quote(row['text_clean'], row['sentiment_label']),
    axis=1
)

print("Extracting medium quotes (100-250 chars)...")
df['quote_medium'] = df.apply(
    lambda row: extract_medium_quote(row['text_clean'], row['sentiment_label']),
    axis=1
)

print("Extracting long quotes (250-500 chars)...")
df['quote_long'] = df.apply(
    lambda row: extract_long_quote(row['text_clean'], row['sentiment_label']),
    axis=1
)

print("Extracting full quotes (up to 1000 chars)...")
df['quote_full'] = df['text_clean'].apply(extract_full_quote)

# Add character counts for each version so the marketing team can filter
df['len_short']  = df['quote_short'].str.len()
df['len_medium'] = df['quote_medium'].str.len()
df['len_long']   = df['quote_long'].str.len()
df['len_full']   = df['quote_full'].str.len()

print("\nExtraction complete.")
print(f"\nAvg quote lengths:")
print(f"  Short:  {df['len_short'].mean():.0f} chars")
print(f"  Medium: {df['len_medium'].mean():.0f} chars")
print(f"  Long:   {df['len_long'].mean():.0f} chars")
print(f"  Full:   {df['len_full'].mean():.0f} chars")

Cleaning text...
Extracting short quotes (under 100 chars)...
Extracting medium quotes (100-250 chars)...
Extracting long quotes (250-500 chars)...
Extracting full quotes (up to 1000 chars)...

Extraction complete.

Avg quote lengths:
  Short:  162 chars
  Medium: 292 chars
  Long:   379 chars
  Full:   803 chars


# Top and Bottom

In [6]:
# For each course extract the top N positive and bottom N negative quotes
# ranked by sentiment compound score.
# Only includes courses with at least 5 reviews.
# Results are stored in a single long-format DataFrame with a quote_type
# column indicating whether each row is a top or bottom quote.

TOP_N    = 10  # number of top positive quotes per course
BOTTOM_N = 5   # number of bottom negative quotes per course
# Note: there are fewer negative reviews overall so BOTTOM_N is smaller

# Columns to include in the output
OUTPUT_COLS = [
    'quote_type', 'course', 'author', 'role', 'source',
    'overall_experience', 'sentiment_compound', 'sentiment_label',
    'quote_short', 'quote_medium', 'quote_long', 'quote_full',
    'len_short', 'len_medium', 'len_long',
]

# Get courses with enough reviews
course_counts = df.dropna(subset=['course'])['course'].value_counts()
valid_courses = course_counts[course_counts >= 5].index.tolist()
print(f"Courses with >= 5 reviews: {valid_courses}")

all_quotes = []

for course in valid_courses:
    course_df = df[df['course'] == course].copy()

    # Top N positive quotes — ranked by compound score descending
    # Deduplicated by author so the same person does not appear multiple times
    top = (
        course_df[course_df['sentiment_label'] == 'positive']
        .sort_values('sentiment_compound', ascending=False)
        .drop_duplicates(subset=['author'])
        .head(TOP_N)
        .copy()
    )
    top['quote_type'] = 'TOP — Positive'
    all_quotes.append(top[OUTPUT_COLS])

    # Bottom N negative + neutral quotes — ranked by compound score ascending
    bottom = (
        course_df[course_df['sentiment_label'].isin(['negative', 'neutral'])]
        .sort_values('sentiment_compound', ascending=True)
        .drop_duplicates(subset=['author'])
        .head(BOTTOM_N)
        .copy()
    )
    bottom['quote_type'] = 'BOTTOM — Negative/Neutral'
    all_quotes.append(bottom[OUTPUT_COLS])

# Also add an overall (all courses) section for general use
overall_top = (
    df[df['sentiment_label'] == 'positive']
    .sort_values('sentiment_compound', ascending=False)
    .drop_duplicates(subset=['author'])
    .head(TOP_N)
    .copy()
)
overall_top['quote_type'] = 'TOP — Positive'
overall_top['course']     = 'All Courses'
all_quotes.append(overall_top[OUTPUT_COLS])

overall_bottom = (
    df[df['sentiment_label'].isin(['negative', 'neutral'])]
    .sort_values('sentiment_compound', ascending=True)
    .drop_duplicates(subset=['author'])
    .head(BOTTOM_N)
    .copy()
)
overall_bottom['quote_type'] = 'BOTTOM — Negative/Neutral'
overall_bottom['course']     = 'All Courses'
all_quotes.append(overall_bottom[OUTPUT_COLS])

# Combine all into one DataFrame
quotes_df = pd.concat(all_quotes, ignore_index=True)

print(f"\nTotal quote rows extracted: {len(quotes_df)}")
print(f"\nBreakdown by course and type:")
print(quotes_df.groupby(['course', 'quote_type']).size().to_string())

Courses with >= 5 reviews: ['Full-Stack Web & App', 'Data Science', 'Product Design', 'Marketing Analytics']

Total quote rows extracted: 58

Breakdown by course and type:
course                quote_type               
All Courses           BOTTOM — Negative/Neutral     5
                      TOP — Positive               10
Data Science          TOP — Positive               10
Full-Stack Web & App  BOTTOM — Negative/Neutral     3
                      TOP — Positive               10
Marketing Analytics   TOP — Positive               10
Product Design        TOP — Positive               10


# Exports

In [7]:
# Export the full quote table to CSV.
# The CSV is structured so the marketing team can:
#   - Filter by course to find quotes for a specific program
#   - Filter by quote_type to see top or bottom quotes
#   - Sort by sentiment_compound to find the strongest quotes
#   - Choose between short, medium, long, or full quote versions
#   - Filter by len_short / len_medium / len_long to find quotes
#     that fit a specific character limit (e.g. a tweet or banner)

csv_path = os.path.join(EXPORT_DIR, 'marketing_quotes_all.csv')
quotes_df.to_csv(csv_path, index=False)

print(f"Saved: marketing_quotes_all.csv")
print(f"Rows: {len(quotes_df)}")
print(f"Columns: {quotes_df.columns.tolist()}")

# Also save a simplified version with just the most useful columns
# for teams who do not need all the metadata
simple_df = quotes_df[[
    'quote_type', 'course', 'author', 'role',
    'overall_experience', 'sentiment_compound',
    'quote_short', 'quote_medium', 'quote_long'
]].copy()

simple_path = os.path.join(EXPORT_DIR, 'marketing_quotes_simple.csv')
simple_df.to_csv(simple_path, index=False)
print(f"Saved: marketing_quotes_simple.csv (simplified version)")

Saved: marketing_quotes_all.csv
Rows: 58
Columns: ['quote_type', 'course', 'author', 'role', 'source', 'overall_experience', 'sentiment_compound', 'sentiment_label', 'quote_short', 'quote_medium', 'quote_long', 'quote_full', 'len_short', 'len_medium', 'len_long']
Saved: marketing_quotes_simple.csv (simplified version)


# Summary

In [8]:
# Generate a formatted Word document presenting all quotes organised
# by course with clear sections for top and bottom quotes.
# Each quote is shown in all three lengths so the marketing team
# can choose the version that fits their format.
#
# The document is styled in WBS brand colours and ready to share
# directly with the marketing team.

# Install docx if not already available
import subprocess
subprocess.run(['npm', 'install', '-g', 'docx'], capture_output=True)

import json

# Build the data structure to pass to the JS script
sections = []
all_courses = ['All Courses'] + valid_courses

for course in all_courses:
    course_data = {'course': course, 'top': [], 'bottom': []}
    course_rows = quotes_df[quotes_df['course'] == course]

    for _, row in course_rows[course_rows['quote_type'] == 'TOP — Positive'].iterrows():
        course_data['top'].append({
            'author':    str(row['author']) if pd.notna(row['author']) else 'Anonymous',
            'role':      str(row['role']) if pd.notna(row['role']) else '',
            'rating':    str(row['overall_experience']) if pd.notna(row['overall_experience']) else '',
            'sentiment': str(round(float(row['sentiment_compound']), 3)),
            'short':     str(row['quote_short']),
            'medium':    str(row['quote_medium']),
            'long':      str(row['quote_long']),
        })

    for _, row in course_rows[course_rows['quote_type'] == 'BOTTOM — Negative/Neutral'].iterrows():
        course_data['bottom'].append({
            'author':    str(row['author']) if pd.notna(row['author']) else 'Anonymous',
            'role':      str(row['role']) if pd.notna(row['role']) else '',
            'rating':    str(row['overall_experience']) if pd.notna(row['overall_experience']) else '',
            'sentiment': str(round(float(row['sentiment_compound']), 3)),
            'short':     str(row['quote_short']),
            'medium':    str(row['quote_medium']),
            'long':      str(row['quote_long']),
        })

    sections.append(course_data)

# Save the data as JSON for the JS script to read
data_path = '/content/quotes_data.json'
with open(data_path, 'w', encoding='utf-8') as f:
    json.dump(sections, f, ensure_ascii=False)

print(f"Data prepared: {len(sections)} course sections")
for s in sections:
    print(f"  {s['course']}: {len(s['top'])} top, {len(s['bottom'])} bottom")

Data prepared: 5 course sections
  All Courses: 10 top, 5 bottom
  Full-Stack Web & App: 10 top, 3 bottom
  Data Science: 10 top, 0 bottom
  Product Design: 10 top, 0 bottom
  Marketing Analytics: 10 top, 0 bottom


In [9]:
# Generate the Word document with all marketing quotes.
# Installs docx locally in /content so Node can find it,
# builds the document from the quotes_data.json file,
# and saves the result to the export directory on Drive.

import subprocess
import os
import shutil
import json

# Install docx locally in /content where the script will run
print("Installing docx...")
result = subprocess.run(
    ['npm', 'install', 'docx'],
    capture_output=True, text=True, cwd='/content'
)
print(result.stdout[-200:] if result.stdout else "")
if result.returncode != 0:
    print("npm install error:", result.stderr[:300])

# Verify installation
check = subprocess.run(
    ['node', '-e', "require('docx'); console.log('docx OK')"],
    capture_output=True, text=True, cwd='/content'
)
print(check.stdout.strip())
if check.returncode != 0:
    print("docx not found:", check.stderr[:200])

js_script = r"""
const {
  Document, Packer, Paragraph, TextRun, Table, TableRow, TableCell,
  HeadingLevel, AlignmentType, BorderStyle, WidthType, ShadingType,
  VerticalAlign, PageBreak, LevelFormat
} = require('docx');
const fs = require('fs');

const data = JSON.parse(fs.readFileSync('/content/quotes_data.json', 'utf8'));

const CORAL       = "E8474F";
const DARK        = "2C2C2C";
const WHITE       = "FFFFFF";
const LIGHT       = "F5F5F5";
const MID         = "DDDDDD";
const MUTED       = "888888";
const CORAL_LIGHT = "FDECEA";
const BLUE_LIGHT  = "EAF4FB";
const BLUE_DARK   = "1A5276";

const border    = { style: BorderStyle.SINGLE, size: 1, color: MID };
const borders   = { top: border, bottom: border, left: border, right: border };
const noBorder  = { style: BorderStyle.NONE, size: 0, color: WHITE };
const noBorders = { top: noBorder, bottom: noBorder, left: noBorder, right: noBorder };

function h1(text) {
  return new Paragraph({
    heading: HeadingLevel.HEADING_1,
    spacing: { before: 400, after: 160 },
    children: [new TextRun({ text, font: "Arial", size: 34, bold: true, color: CORAL })]
  });
}

function h2(text, color = DARK) {
  return new Paragraph({
    spacing: { before: 240, after: 100 },
    border: { bottom: { style: BorderStyle.SINGLE, size: 3, color: color, space: 3 } },
    children: [new TextRun({ text, font: "Arial", size: 24, bold: true, color: color })]
  });
}

function body(text) {
  return new Paragraph({
    spacing: { after: 100 },
    children: [new TextRun({ text, font: "Arial", size: 20, color: DARK })]
  });
}

function spacer(n = 120) {
  return new Paragraph({
    spacing: { after: n },
    children: [new TextRun({ text: "" })]
  });
}

function metaLine(author, role, rating, sentiment) {
  const parts = [];
  if (author)    parts.push(author);
  if (role)      parts.push(role);
  if (rating)    parts.push("Rating: " + rating + " / 5");
  if (sentiment) parts.push("Sentiment: " + sentiment);
  return new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({
      text: parts.join("  |  "),
      font: "Arial", size: 17, bold: true, color: MUTED
    })]
  });
}

function quoteVersions(q, isPositive) {
  const accent = isPositive ? CORAL : BLUE_DARK;
  const shade  = isPositive ? CORAL_LIGHT : BLUE_LIGHT;

  function quoteRow(labelText, quoteText) {
    return [
      new TableRow({
        children: [
          new TableCell({
            borders: noBorders,
            width: { size: 180, type: WidthType.DXA },
            shading: { fill: accent, type: ShadingType.CLEAR },
            children: [new Paragraph({ children: [new TextRun({ text: "" })] })]
          }),
          new TableCell({
            borders: noBorders,
            width: { size: 8846, type: WidthType.DXA },
            shading: { fill: shade, type: ShadingType.CLEAR },
            margins: { top: 100, bottom: 100, left: 200, right: 160 },
            children: [
              new Paragraph({
                spacing: { after: 40 },
                children: [new TextRun({
                  text: labelText,
                  font: "Arial", size: 15, bold: true, color: accent, allCaps: true
                })]
              }),
              new Paragraph({
                spacing: { after: 0 },
                children: [new TextRun({
                  text: "\u201C" + quoteText + "\u201D",
                  font: "Arial", size: 20, italics: true, color: DARK
                })]
              })
            ]
          })
        ]
      }),
      new TableRow({
        children: [
          new TableCell({
            borders: noBorders,
            width: { size: 180, type: WidthType.DXA },
            shading: { fill: WHITE, type: ShadingType.CLEAR },
            children: [new Paragraph({ spacing: { after: 60 }, children: [new TextRun({ text: "" })] })]
          }),
          new TableCell({
            borders: noBorders,
            width: { size: 8846, type: WidthType.DXA },
            shading: { fill: WHITE, type: ShadingType.CLEAR },
            children: [new Paragraph({ spacing: { after: 60 }, children: [new TextRun({ text: "" })] })]
          })
        ]
      })
    ];
  }

  return new Table({
    width: { size: 9026, type: WidthType.DXA },
    columnWidths: [180, 8846],
    rows: [
      ...quoteRow("SHORT", q.short),
      ...quoteRow("MEDIUM", q.medium),
      ...quoteRow("LONG", q.long),
    ]
  });
}

const children = [
  // Cover page
  new Paragraph({
    spacing: { before: 1440, after: 120 },
    children: [new TextRun({ text: "WBS Coding School", font: "Arial", size: 48, bold: true, color: CORAL })]
  }),
  new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({ text: "Marketing Quote Bank", font: "Arial", size: 40, bold: true, color: DARK })]
  }),
  new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({
      text: "Top and Bottom Reviews by Course \u2014 Short, Medium, and Long Versions",
      font: "Arial", size: 24, color: MUTED
    })]
  }),
  new Paragraph({
    spacing: { after: 600 },
    children: [new TextRun({
      text: "Prepared by Essentis  |  March 2026  |  For internal marketing use",
      font: "Arial", size: 18, color: MUTED
    })]
  }),

  // How to use section
  new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({ text: "How to use this document", font: "Arial", size: 22, bold: true, color: DARK })]
  }),
  new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({
      text: "Each review is presented in three lengths so you can choose the version that fits your format:",
      font: "Arial", size: 20, color: DARK
    })]
  }),
  new Paragraph({
    spacing: { after: 60 },
    children: [new TextRun({
      text: "SHORT     under 100 characters \u2014 one punchy sentence, ideal for social media, banners, and pull quotes",
      font: "Arial", size: 19, color: DARK
    })]
  }),
  new Paragraph({
    spacing: { after: 60 },
    children: [new TextRun({
      text: "MEDIUM    100\u2013250 characters \u2014 two to three sentences, ideal for website testimonials and ads",
      font: "Arial", size: 19, color: DARK
    })]
  }),
  new Paragraph({
    spacing: { after: 160 },
    children: [new TextRun({
      text: "LONG      250\u2013500 characters \u2014 full paragraph, ideal for case studies and detailed testimonial pages",
      font: "Arial", size: 19, color: DARK
    })]
  }),
  new Paragraph({
    spacing: { after: 80 },
    children: [new TextRun({
      text: "Positive quotes are highlighted in coral red. Critical quotes are highlighted in blue and are included for balance and product improvement context.",
      font: "Arial", size: 19, italics: true, color: MUTED
    })]
  }),
];

// Build one section per course
data.forEach((section) => {
  children.push(new Paragraph({ children: [new PageBreak()] }));
  children.push(h1(section.course));

  if (section.top.length > 0) {
    children.push(h2("Top Positive Reviews", CORAL));
    children.push(spacer(80));
    section.top.forEach((q) => {
      children.push(metaLine(q.author, q.role, q.rating, q.sentiment));
      children.push(quoteVersions(q, true));
      children.push(spacer(200));
    });
  }

  if (section.bottom.length > 0) {
    children.push(h2("Critical Reviews", BLUE_DARK));
    children.push(body(
      "These reviews represent the most critical feedback for this course. " +
      "Included for balance and product improvement context."
    ));
    children.push(spacer(80));
    section.bottom.forEach((q) => {
      children.push(metaLine(q.author, q.role, q.rating, q.sentiment));
      children.push(quoteVersions(q, false));
      children.push(spacer(200));
    });
  }
});

const doc = new Document({
  styles: {
    default: {
      document: { run: { font: "Arial", size: 20, color: DARK } }
    },
    paragraphStyles: [
      {
        id: "Heading1", name: "Heading 1", basedOn: "Normal",
        next: "Normal", quickFormat: true,
        run: { size: 34, bold: true, font: "Arial", color: CORAL },
        paragraph: { spacing: { before: 400, after: 160 }, outlineLevel: 0 }
      }
    ]
  },
  sections: [{
    properties: {
      page: {
        size: { width: 11906, height: 16838 },
        margin: { top: 1260, right: 1260, bottom: 1260, left: 1260 }
      }
    },
    children
  }]
});

Packer.toBuffer(doc).then(buf => {
  fs.writeFileSync('/content/marketing_quotes.docx', buf);
  console.log('Done');
});
"""

# Write the JS script
js_path = '/content/quotes_doc.js'
with open(js_path, 'w', encoding='utf-8') as f:
    f.write(js_script)

# Run the JS script from /content so it finds the local node_modules
result = subprocess.run(
    ['node', js_path],
    capture_output=True, text=True, cwd='/content'
)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr[:500])

# Copy to Drive export directory
if os.path.exists('/content/marketing_quotes.docx'):
    drive_doc_path = os.path.join(EXPORT_DIR, 'marketing_quotes.docx')
    shutil.copy('/content/marketing_quotes.docx', drive_doc_path)
    size_kb = os.path.getsize(drive_doc_path) // 1024
    print(f"Saved: marketing_quotes.docx ({size_kb} KB)")
    print(f"Path: {drive_doc_path}")
else:
    print("ERROR: marketing_quotes.docx was not created. Check the errors above.")

Installing docx...

added 22 packages in 2s

1 package is looking for funding
  run `npm fund` for details

docx OK
Done

Saved: marketing_quotes.docx (30 KB)
Path: /content/drive/Shareddrives/essentis_intern_drive/charts/marketing_quotes/marketing_quotes.docx


In [10]:
# Print a summary of what was exported and how to use the files.

print("=" * 70)
print("MARKETING QUOTE EXTRACTION COMPLETE")
print("=" * 70)
print(f"\nExport directory: {EXPORT_DIR}")
print(f"\nFiles saved:")
for f in sorted(os.listdir(EXPORT_DIR)):
    size = os.path.getsize(os.path.join(EXPORT_DIR, f))
    print(f"  {f}  ({size // 1024} KB)")

print(f"\nTotal quotes extracted: {len(quotes_df)}")
print(f"\nBreakdown:")
print(quotes_df.groupby(['course', 'quote_type']).size().to_string())

print(f"""
HOW TO USE
----------
marketing_quotes.docx
  Formatted Word document organised by course.
  Each review shows SHORT, MEDIUM, and LONG versions.
  Share directly with the marketing team.
  Positive quotes in coral red, critical quotes in blue.

marketing_quotes_all.csv
  Full dataset with all metadata and all quote versions.
  Filter by course, quote_type, sentiment_compound, or len_short
  to find quotes that fit a specific format or character limit.

marketing_quotes_simple.csv
  Simplified version with just the essential columns.
  Easier to browse for teams who do not need the full metadata.

QUOTE LENGTH GUIDE
  short:   under 100 chars  — social media, banners, pull quotes
  medium:  100-250 chars    — website testimonials, ads
  long:    250-500 chars    — case studies, detailed testimonial pages
  full:    up to 1000 chars — blog posts, detailed marketing materials
""")

MARKETING QUOTE EXTRACTION COMPLETE

Export directory: /content/drive/Shareddrives/essentis_intern_drive/charts/marketing_quotes

Files saved:
  marketing_quotes.docx  (30 KB)
  marketing_quotes_all.csv  (235 KB)
  marketing_quotes_simple.csv  (73 KB)
  quotes_all_courses_critical.png  (151 KB)
  quotes_all_courses_positive.png  (649 KB)
  quotes_data_science_positive.png  (479 KB)
  quotes_full-stack_web_and_app_critical.png  (93 KB)
  quotes_full-stack_web_and_app_positive.png  (673 KB)
  quotes_marketing_analytics_positive.png  (390 KB)
  quotes_product_design_positive.png  (460 KB)

Total quotes extracted: 58

Breakdown:
course                quote_type               
All Courses           BOTTOM — Negative/Neutral     5
                      TOP — Positive               10
Data Science          TOP — Positive               10
Full-Stack Web & App  BOTTOM — Negative/Neutral     3
                      TOP — Positive               10
Marketing Analytics   TOP — Positive             

In [11]:
# Generate a visual grid of the top short quotes per course.
# Each course gets its own chart showing the best short quotes
# as styled cards arranged in a grid.
# Charts are saved to the EXPORT_DIR as PNG files.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import textwrap

CORAL_MPL = "#E8474F"
DARK_MPL  = "#2C2C2C"
LIGHT_MPL = "#F5F5F5"
MUTED_MPL = "#888888"
WHITE_MPL = "#FFFFFF"

def wrap_quote(text: str, width: int = 40) -> str:
    """
    Wrap a short quote at word boundaries for display in a chart card.

    Args:
        text (str): Short quote text.
        width (int): Maximum characters per line before wrapping.

    Returns:
        str: Quote with newlines inserted for clean card display.
    """
    return "\n".join(textwrap.wrap(str(text), width=width))


def make_quote_grid(
    quotes: list,
    title: str,
    quote_type: str,
    save_path: str,
    cols: int = 2
) -> None:
    """
    Render a grid of quote cards as a matplotlib figure and save as PNG.

    Each card shows the short quote text, reviewer name, and star rating.
    Positive quotes use the coral WBS brand colour. Critical quotes use
    a dark blue accent. The grid automatically adjusts row count based
    on the number of quotes provided.

    Args:
        quotes (list): List of dicts with keys: short, author, rating, sentiment.
        title (str): Chart title — typically the course name.
        quote_type (str): "positive" or "negative" — controls card colour.
        save_path (str): Full file path to save the PNG.
        cols (int): Number of columns in the grid. Default 2.

    Returns:
        None. Saves the figure directly to save_path.
    """
    if not quotes:
        print(f"  No quotes to render for {title} ({quote_type})")
        return

    accent    = CORAL_MPL if quote_type == 'positive' else "#1A5276"
    card_bg   = "#FDECEA" if quote_type == 'positive' else "#EAF4FB"
    n         = len(quotes)
    rows      = (n + cols - 1) // cols
    fig_w     = 14
    card_h    = 2.4
    header_h  = 1.2
    fig_h     = header_h + rows * (card_h + 0.3) + 0.6

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor(WHITE_MPL)

    # Title bar
    title_ax = fig.add_axes([0, 1 - header_h / fig_h, 1, header_h / fig_h])
    title_ax.set_facecolor(accent)
    title_ax.set_xlim(0, 1)
    title_ax.set_ylim(0, 1)
    title_ax.axis('off')

    type_label = "Top Positive Quotes" if quote_type == 'positive' else "Critical Quotes"
    title_ax.text(
        0.03, 0.65, title,
        fontsize=20, fontweight='bold', color=WHITE_MPL,
        va='center', ha='left', fontfamily='sans-serif'
    )
    title_ax.text(
        0.03, 0.25, type_label + "  —  Short version (under 100 characters)",
        fontsize=11, color=WHITE_MPL, alpha=0.85,
        va='center', ha='left', fontfamily='sans-serif'
    )

    # Quote cards
    pad_l   = 0.03
    pad_r   = 0.03
    pad_top = header_h / fig_h
    gap_x   = 0.02
    gap_y   = 0.3 / fig_h
    card_w  = (1 - pad_l - pad_r - gap_x * (cols - 1)) / cols
    card_h_norm = card_h / fig_h

    for i, q in enumerate(quotes):
        row = i // cols
        col = i % cols

        x = pad_l + col * (card_w + gap_x)
        y = 1 - pad_top - (row + 1) * (card_h_norm + gap_y) + gap_y

        ax = fig.add_axes([x, y, card_w, card_h_norm])
        ax.set_facecolor(card_bg)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')

        # Accent left border
        ax.add_patch(FancyBboxPatch(
            (0, 0), 0.018, 1,
            boxstyle="square,pad=0",
            facecolor=accent, edgecolor='none'
        ))

        # Quote text
        wrapped = wrap_quote(str(q.get('short', '')), width=48)
        ax.text(
            0.04, 0.72,
            f"\u201C{wrapped}\u201D",
            fontsize=10.5, fontstyle='italic', color=DARK_MPL,
            va='top', ha='left', fontfamily='sans-serif',
            wrap=False
        )

        # Attribution line
        author  = str(q.get('author', 'Anonymous'))
        rating  = q.get('rating', '')
        role    = q.get('role', '')
        attr_parts = [author]
        if role and role not in ('nan', ''):
            attr_parts.append(role)
        if rating and str(rating) not in ('nan', ''):
            try:
                rating_float = float(rating)
                rating_int   = int(rating_float)
                stars = '★' * rating_int if rating_float == rating_int else f"{rating_float:.1f} stars"
                attr_parts.append(stars)
            except (ValueError, TypeError):
                pass
        attr = "  |  ".join(attr_parts)

        ax.text(
            0.04, 0.12,
            attr,
            fontsize=9, fontweight='bold', color=MUTED_MPL,
            va='bottom', ha='left', fontfamily='sans-serif'
        )

        # Card border
        for spine_side in ['top', 'bottom', 'left', 'right']:
            ax.spines[spine_side].set_visible(False)
        ax.add_patch(FancyBboxPatch(
            (0.001, 0.01), 0.998, 0.98,
            boxstyle="square,pad=0",
            facecolor='none',
            edgecolor='#E0E0E0', linewidth=0.8
        ))

    # Footer
    footer_ax = fig.add_axes([0, 0, 1, 0.04])
    footer_ax.set_facecolor(LIGHT_MPL)
    footer_ax.set_xlim(0, 1)
    footer_ax.set_ylim(0, 1)
    footer_ax.axis('off')
    footer_ax.text(
        0.03, 0.5,
        "WBS Coding School  |  Essentis Internship Project  |  March 2026  |  For internal marketing use",
        fontsize=8, color=MUTED_MPL, va='center', ha='left', fontfamily='sans-serif'
    )

    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=WHITE_MPL)
    plt.close()
    print(f"  Saved: {os.path.basename(save_path)}")


# -------------------------------------------------------------------------
# Generate charts for each course — positive and critical separately
# -------------------------------------------------------------------------
print("Generating short quote charts...\n")

all_courses_in_order = ['All Courses'] + valid_courses

for course in all_courses_in_order:
    course_rows = quotes_df[quotes_df['course'] == course]
    course_slug = course.lower().replace(' ', '_').replace('&', 'and').replace('/', '_')

    # Positive chart
    top_rows = course_rows[course_rows['quote_type'] == 'TOP — Positive']
    if len(top_rows) > 0:
        top_quotes_list = top_rows[[
            'quote_short', 'author', 'overall_experience', 'role', 'sentiment_compound'
        ]].rename(columns={
            'quote_short': 'short',
            'overall_experience': 'rating',
            'sentiment_compound': 'sentiment'
        }).to_dict('records')

        save_path = os.path.join(EXPORT_DIR, f'quotes_{course_slug}_positive.png')
        make_quote_grid(
            quotes     = top_quotes_list,
            title      = course,
            quote_type = 'positive',
            save_path  = save_path,
            cols       = 2
        )

    # Critical chart
    bottom_rows = course_rows[course_rows['quote_type'] == 'BOTTOM — Negative/Neutral']
    if len(bottom_rows) > 0:
        bottom_quotes_list = bottom_rows[[
            'quote_short', 'author', 'overall_experience', 'role', 'sentiment_compound'
        ]].rename(columns={
            'quote_short': 'short',
            'overall_experience': 'rating',
            'sentiment_compound': 'sentiment'
        }).to_dict('records')

        save_path = os.path.join(EXPORT_DIR, f'quotes_{course_slug}_critical.png')
        make_quote_grid(
            quotes     = bottom_quotes_list,
            title      = course,
            quote_type = 'negative',
            save_path  = save_path,
            cols       = 2
        )

# -------------------------------------------------------------------------
# Print summary of all files saved
# -------------------------------------------------------------------------
print(f"\nAll charts saved to: {EXPORT_DIR}")
print("\nFiles in export directory:")
for f in sorted(os.listdir(EXPORT_DIR)):
    size_kb = os.path.getsize(os.path.join(EXPORT_DIR, f)) // 1024
    print(f"  {f}  ({size_kb} KB)")

Generating short quote charts...

  Saved: quotes_all_courses_positive.png
  Saved: quotes_all_courses_critical.png
  Saved: quotes_full-stack_web_and_app_positive.png
  Saved: quotes_full-stack_web_and_app_critical.png
  Saved: quotes_data_science_positive.png
  Saved: quotes_product_design_positive.png
  Saved: quotes_marketing_analytics_positive.png

All charts saved to: /content/drive/Shareddrives/essentis_intern_drive/charts/marketing_quotes

Files in export directory:
  marketing_quotes.docx  (30 KB)
  marketing_quotes_all.csv  (235 KB)
  marketing_quotes_simple.csv  (73 KB)
  quotes_all_courses_critical.png  (151 KB)
  quotes_all_courses_positive.png  (649 KB)
  quotes_data_science_positive.png  (479 KB)
  quotes_full-stack_web_and_app_critical.png  (93 KB)
  quotes_full-stack_web_and_app_positive.png  (673 KB)
  quotes_marketing_analytics_positive.png  (390 KB)
  quotes_product_design_positive.png  (460 KB)
